# RHIC Upgrade Group Notebook: Filling the Gap Between RHIC and LHC
## Probing a New Slice of QGP Phase Space

**Vanderbilt Programs for Talented Youth | Instructor: Jennifer James, Ph.D. Candidate**

---

### The challenge

RHIC collides gold nuclei at up to sqrt(s_NN) = 200 GeV per nucleon pair. The LHC collides lead nuclei starting at sqrt(s_NN) = 2760 GeV, over ten times higher. Between those two numbers is a huge, largely unexplored gap: no heavy-ion collider has ever run there.

Your group's project is not about building a new detector subsystem. It is about asking whether RHIC itself, its accelerator, could be pushed to reach into that gap, and what new QGP physics might be sitting there waiting to be measured.

In this notebook, you will look at how a key QGP observable changes with collision energy, identify the size of the unexplored gap, and calculate what it would actually take, in terms of accelerator hardware, to reach into it.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

print("Libraries loaded!")

---

## Part 1: An Excitation Function Across Collider Energies

A good way to track how the QGP fireball changes with collision energy is to look at **charged particle multiplicity**, roughly how many particles come flying out per participating nucleon pair, in the most central (head-on) collisions. This quantity has been measured across many different accelerators.

The values below are approximate, illustrative numbers based on published trends from AGS, SPS, RHIC, and LHC heavy-ion measurements, not exact published figures.

In [ ]:
# ── Approximate central dN_ch/deta per (0.5 * Npart), by facility ──────────────
facility   = np.array(['AGS', 'SPS', 'RHIC-62', 'RHIC-130', 'RHIC-200', 'LHC-2760', 'LHC-5020'])
sqrtsNN    = np.array([4.8, 17.3, 62.4, 130.0, 200.0, 2760.0, 5020.0])   # GeV
multiplicity = np.array([1.0, 2.1, 2.9, 3.3, 3.6, 8.3, 8.9])            # dNch/deta per 0.5*Npart
mult_err   = multiplicity * 0.05   # illustrative 5% uncertainty

def power_law(sqrts, A, n):
    return A * sqrts**n

popt, pcov = curve_fit(power_law, sqrtsNN, multiplicity, p0=[1.0, 0.15])
A_fit, n_fit = popt

sqrts_fine = np.logspace(np.log10(3), np.log10(8000), 300)
fit_curve  = power_law(sqrts_fine, A_fit, n_fit)

fig, ax = plt.subplots(figsize=(9, 6))
ax.errorbar(sqrtsNN, multiplicity, yerr=mult_err, fmt='o', markersize=8,
            color='black', capsize=4, label='Existing measurements', zorder=5)
ax.plot(sqrts_fine, fit_curve, color='steelblue', linewidth=2,
        label=f'Power-law fit: multiplicity ~ (sqrt(s_NN))^{n_fit:.2f}')
for x, y, name in zip(sqrtsNN, multiplicity, facility):
    ax.annotate(name, (x, y), textcoords='offset points', xytext=(6, 6), fontsize=8)
ax.axvspan(200, 2760, alpha=0.12, color='gold', label='Unexplored gap: RHIC to LHC')
ax.set_xscale('log')
ax.set_xlabel(r'$\sqrt{s_{NN}}$ (GeV)', fontsize=12)
ax.set_ylabel(r'Central $dN_{ch}/d\eta$ per (0.5 $N_{part}$)', fontsize=12)
ax.set_title('Charged particle multiplicity vs collision energy', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print(f"Fitted scaling: multiplicity ~ (sqrt(s_NN))^{n_fit:.2f}")
print("Notice the shaded gap: nearly a factor of 14 in energy with no heavy-ion data at all.")

---

## Part 2: What Would an Intermediate Energy Point Tell Us?

Suppose RHIC could be upgraded to reach sqrt(s_NN) = 1000 GeV, right in the middle of that gap. The smooth fit predicts one value, but the fit was built entirely from points *outside* the gap. If the real measurement disagreed with the smooth extrapolation, that would be new physics: onset of new dynamics (like gluon saturation effects) that only appear at intermediate energies and are invisible in a simple power-law trend.

In [ ]:
target_energy = 1000.0  # GeV, a representative upgrade target inside the gap
predicted_mult = power_law(target_energy, A_fit, n_fit)

print(f"Smooth extrapolation predicts multiplicity = {predicted_mult:.2f} at sqrt(s_NN) = {target_energy} GeV")
print()
print("A real measurement at this energy could:")
print("  - Confirm the smooth trend, ruling out new physics in this energy range")
print("  - Or reveal a deviation, pointing to new QGP dynamics not visible at RHIC or LHC alone")
print()
print("Either outcome is valuable. Right now, nobody knows which one is true, because no")
print("collider has ever measured a heavy-ion collision in this energy range.")

---

## Part 3: Can RHIC Actually Reach This Energy?

RHIC is a fixed-size ring, roughly 3.8 km in circumference. Just like the cyclotron and the general accelerator physics you have already covered, the maximum momentum a ring can hold onto is set by:

$$p = qBr$$

Since the ring's overall radius is fixed by existing tunnels and infrastructure, reaching higher energy means increasing the magnetic field strength of the bending (dipole) magnets around the ring, rather than making the ring physically bigger.

In [ ]:
# ── Simplified single-magnet approximation of the RHIC ring ─────────────────────
# p[GeV/c] = 0.2998 * B[T] * r[m], for a singly-charged particle

def field_for_momentum(p_GeV, r_m):
    return p_GeV / (0.2998 * r_m)

def momentum_for_field(B_T, r_m):
    return 0.2998 * B_T * r_m

# ── Back out an effective bending radius consistent with RHIC's known operating point ──
current_energy_GeV = 100.0   # GeV per beam, current RHIC top energy (protons)
current_field_T     = 3.45    # tesla, approximate RHIC dipole field at top energy
r_effective = current_energy_GeV / (0.2998 * current_field_T)

print(f"Effective bending radius consistent with RHIC's current operating point: {r_effective:.1f} m")
print(f"(This is an illustrative single-magnet approximation, not the full ring geometry.)")
print()

# ── What field would be needed for the target energy? ─────────────────────────
target_energy_per_beam = target_energy / 2   # sqrt(s_NN) split between two counter-rotating beams
B_needed = field_for_momentum(target_energy_per_beam, r_effective)

print(f"To reach sqrt(s_NN) = {target_energy:.0f} GeV (i.e. {target_energy_per_beam:.0f} GeV per beam):")
print(f"  Required dipole field: {B_needed:.2f} T")
print(f"  Current RHIC dipole field: {current_field_T:.2f} T")
print(f"  Increase needed: {B_needed/current_field_T:.1f}x stronger magnets")

In [ ]:
target_energies = np.linspace(200, 2760, 100)
fields_needed = field_for_momentum(target_energies/2, r_effective)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(target_energies, fields_needed, color='steelblue', linewidth=2)
ax.axhline(y=current_field_T, color='firebrick', linestyle='--', linewidth=1.5,
           label=f'Current RHIC field ({current_field_T} T)')
ax.axvline(x=200, color='gray', linestyle=':', linewidth=1.2, label='Current RHIC energy (200 GeV)')
ax.axvline(x=target_energy, color='gold', linestyle=':', linewidth=1.5,
           label=f'Target upgrade energy ({target_energy:.0f} GeV)')
ax.set_xlabel(r'Target $\sqrt{s_{NN}}$ (GeV)', fontsize=12)
ax.set_ylabel('Required dipole field (T)', fontsize=12)
ax.set_title('Magnet field needed to reach higher collision energy\n(fixed ring radius)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Part 4: Would the Detectors Also Need to Change?

Higher collision energy does not just mean a stronger accelerator magnet. It changes what comes out of the collisions, and that has consequences for any detector sitting around the interaction point (sPHENIX included).

- **Calorimeters** need to be thicker: higher-energy particles produce deeper showers, and a calorimeter that fully contained a 200 GeV/nucleon-pair collision's particles might let higher-energy particles punch through
- **Tracking** needs better momentum resolution: higher-momentum tracks curve less, so distinguishing them requires either a stronger tracker magnet or a larger tracker radius
- **Readout electronics** need to handle higher particle multiplicities and rates without pile-up
- **Shielding** may need to be reconsidered, since higher-energy collisions can produce more penetrating secondary radiation

---

## Your Group's Checkpoint Questions

1. In Part 1, why is the gap between RHIC and LHC energies scientifically interesting, rather than just an engineering curiosity? What could be hiding in that gap that a smooth extrapolation would miss?

2. In Part 3, we held the ring radius fixed and solved for the magnetic field needed. Compare this to the LBL cyclotron group's approach, where they are holding the field fixed and increasing the radius instead. Why might increasing magnet field be more realistic for RHIC than digging a bigger tunnel?

3. Look at the field values you calculated in Part 3. RHIC's current magnets are already superconducting. What does the size of the required field increase tell you about how hard (or easy) this upgrade would be in practice?

4. In Part 4, pick one detector subsystem (calorimeter, tracker, or shielding) and explain, in your own words, why higher collision energy specifically requires a change to that subsystem.

5. If you only had budget to either double the magnet field or double the tracker's momentum resolution, which would get you closer to actually being able to measure new physics in the RHIC-LHC gap? Defend your answer.